In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.v2 as transforms

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
)

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:3]: 
        print(os.path.join(dirname, filename))

In [ ]:
SOURCE_NOTEBOOK_DIR = "/kaggle/input/datasets/rijalulghodi/aptos-processed/aptos_data"

BALANCED_DIR = os.path.join(SOURCE_NOTEBOOK_DIR, "train_balanced")
VAL_TEST_DIR = os.path.join(SOURCE_NOTEBOOK_DIR, "val_test")
AUGMENTED_DIR = os.path.join(SOURCE_NOTEBOOK_DIR, "augmented_raw")
METADATA_DIR = os.path.join(SOURCE_NOTEBOOK_DIR, "csv_metadata")

for name, path in [("BALANCED_DIR", BALANCED_DIR), ("VAL_TEST_DIR", VAL_TEST_DIR), ("AUGMENTED_DIR", AUGMENTED_DIR), 
                   ("METADATA_DIR", METADATA_DIR)]:
    if os.path.exists(path):
        n_files = len(os.listdir(path))
        print(f"{name}: {path} -> {n_files} file ditemukan")
    else:
        print(f"PERINGATAN: {name} tidak ditemukan di {path}")

In [ ]:
balanced_df = pd.read_csv(os.path.join(METADATA_DIR, 'balanced_train_data.csv'))
# balanced_df = balanced_df[~balanced_df['id_code'].str.contains('_aug', na=False)].reset_index(drop=True)
print("=== PENYARINGAN DATA ASLI (TIMPANG) ===")
print(f"Jumlah baris setelah filter: {len(balanced_df)}")

print("\nDistribusi kelas data asli yang timpang:")
print(balanced_df['diagnosis'].value_counts().sort_index())

val_df = pd.read_csv(os.path.join(METADATA_DIR, 'val_data.csv'))
test_df = pd.read_csv(os.path.join(METADATA_DIR, 'test_data.csv'))

# 3. Verifikasi jumlah data yang berhasil dimuat
print(f"Jumlah data balanced_df (Train): {len(balanced_df)}")
print(f"Jumlah data val_df (Validation) : {len(val_df)}")
print(f"Jumlah data test_df (Test)      : {len(test_df)}")

# Tampilkan 5 baris pertama dari balanced_df
balanced_df.head()

In [ ]:
sample_row = val_df.iloc[0]
ben_graham_path = os.path.join(VAL_TEST_DIR, f"{sample_row['id_code']}.png")

ben_img = cv2.imread(ben_graham_path)
ben_img = cv2.cvtColor(ben_img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 1, figsize=(12, 6))
axes.imshow(ben_img)
axes.set_title(f"Contoh data\nID: {sample_row['id_code']}")
axes.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
train_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

In [ ]:
val_test_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
class APTOSDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = f"{row['id_code']}.png"
        img_path = os.path.join(self.img_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row['diagnosis'], dtype=torch.float32)
        return image, label

In [ ]:
train_dataset = APTOSDataset(balanced_df, BALANCED_DIR, transform=train_transform)
val_dataset = APTOSDataset(val_df, VAL_TEST_DIR, transform=val_test_transform)
test_dataset = APTOSDataset(test_df, VAL_TEST_DIR, transform=val_test_transform)

In [ ]:
class_counts = balanced_df['diagnosis'].value_counts().sort_index()
class_weights_sampler = 1.0 / class_counts
sample_weights = balanced_df['diagnosis'].map(class_weights_sampler).values
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True, num_workers=2, persistent_workers=True)
# train_loader = DataLoader(train_dataset, batch_size=64, pin_memory=True, sampler=sampler, num_workers=2, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, pin_memory=True, num_workers=2, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True, num_workers=2, persistent_workers=True)

images, labels = next(iter(train_loader))
print(f"Shape batch gambar (PyTorch format [B, C, H, W]): {images.shape}")
print(f"Shape batch label: {labels.shape}")

# Inisiasi Model

In [ ]:
# ==========================================================
# Sel 1 — Setup: clone repo ViT-5 & install dependency
# (Pastikan "Internet" diaktifkan di pengaturan notebook Kaggle)
# ==========================================================
!git clone https://github.com/wangf3014/ViT-5.git /kaggle/working/ViT-5
!pip install -q einops

# ==========================================================
# Sel 2 — Registrasi arsitektur ViT-5 ke timm
# ViT-5 TIDAK ada di hub resmi timm, jadi arsitekturnya (vit5_small/
# vit5_base/vit5_large) harus didaftarkan manual dengan mengimpor
# models_vit5.py dari repo yang baru di-clone (memakai @register_model
# milik timm secara internal).
# ==========================================================
import sys
sys.path.append("/kaggle/working/ViT-5")

import timm
import models_vit5  # noqa: F401  -> import ini WAJIB, efek sampingnya mendaftarkan model vit5_* ke timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# ==========================================================
# Sel 3 — Download checkpoint ImageNet-1k pretrained
# Pilih salah satu varian sesuai kebutuhan (Small/Base/Large, res 224/384)
# ==========================================================
CHECKPOINT_URL = "https://huggingface.co/FengWang3211/ViT-5/resolve/main/vit5_base_patch16_224.pth"
CHECKPOINT_PATH = "/kaggle/working/vit5_base_patch16_224.pth"

!wget -q -O {CHECKPOINT_PATH} {CHECKPOINT_URL}
print("Checkpoint berhasil diunduh.")

# ==========================================================
# Sel 4 — Inisialisasi model ViT-5 + load bobot pretrained
# ==========================================================
NUM_CLASSES = 1  
MODEL_NAME = "vit5_base"  

In [ ]:
# 1) Buat model dengan num_classes bawaan checkpoint (ImageNet-1k = 1000)
#    supaya bentuk head cocok saat load_state_dict, baru diganti belakangan.
model = timm.create_model(
    MODEL_NAME,
    pretrained=False,   # bobot TIDAK diambil otomatis dari hub timm -> load manual di bawah
    num_classes=1000,
    drop_path_rate=0.25,
)

# 2) Load checkpoint dan sesuaikan formatnya
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

# Beberapa checkpoint membungkus state_dict di dalam key "model" atau "model_ema"
if "model" in checkpoint:
    state_dict = checkpoint["model"]
elif "model_ema" in checkpoint:
    state_dict = checkpoint["model_ema"]
else:
    state_dict = checkpoint

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys : {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")

# 3) Baru ganti head klasifikasi ke NUM_CLASSES yang kamu butuhkan
model.reset_classifier(num_classes=NUM_CLASSES)

if torch.cuda.device_count() > 1:
    print(f"=== Menggunakan {torch.cuda.device_count()} GPU T4 Sekaligus! ===")
    # Bungkus model agar otomatis membagi beban ke GPU [0] dan [1]
    model = nn.DataParallel(model)

model = model.to(device)
print("Model ViT-5 berhasil diinisialisasi!")


In [ ]:
for param in model.module.parameters():
    param.requires_grad = False

for param in model.module.head.parameters():
    param.requires_grad = True

In [ ]:
from torchinfo import summary

summary(
    model.module, 
    input_size=(16, 3, 224, 224), 
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} / Total: {total_params:,} ({trainable_params/total_params*100:.2f}%)")

In [ ]:
criterion = nn.MSELoss()
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

In [ ]:
def get_logits(outputs):
    """timm mengembalikan tensor logits langsung, berbeda dengan
    HF transformers yang mengembalikan objek dengan atribut .logits."""
    return outputs

def regression_to_class(outputs, num_classes=5):
    """Konversi output regresi kontinu menjadi label kelas diskrit (0..num_classes-1)."""
    return torch.round(torch.clamp(outputs, 0, num_classes - 1))

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch, total_epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{total_epochs} [Train]", leave=True)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        if scaler is not None:
            with torch.amp.autocast('cuda'):
                outputs = get_logits(model(images)).squeeze(-1)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = get_logits(model(images)).squeeze(-1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        
        preds = regression_to_class(outputs)
        
        correct_predictions += torch.sum(preds == labels)
        total_samples += labels.size(0)

        current_loss = running_loss / total_samples
        current_acc = (correct_predictions.float() / total_samples).item()
        pbar.set_postfix({"mse_loss": f"{current_loss:.4f}", "acc": f"{current_acc:.4f}"})

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions.float() / total_samples).item()
    
    return epoch_loss, epoch_acc

In [ ]:
def validate_one_epoch(model, dataloader, criterion, device, epoch, total_epochs):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_preds, all_labels = [], []

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{total_epochs} [Val]  ", leave=True)

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            # Evaluasi dengan AMP Autocast agar komputasi di val set juga cepat
            if torch.cuda.is_available():
                with torch.amp.autocast('cuda'):
                    outputs = get_logits(model(images)).squeeze(-1)
            else:
                outputs = get_logits(model(images)).squeeze(-1)

            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            preds = regression_to_class(outputs)
            
            correct_predictions += torch.sum(preds == labels)
            total_samples += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            current_loss = running_loss / total_samples
            current_acc = (correct_predictions.float() / total_samples).item()
            pbar.set_postfix({"mse_loss": f"{current_loss:.4f}", "acc": f"{current_acc:.4f}"})

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions.float() / total_samples).item()
    epoch_qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
    
    return epoch_loss, epoch_acc, epoch_qwk

In [ ]:
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_qwk': []
}

In [ ]:
best_val_qwk = -1.0
best_val_loss = float('inf')
patience = 10 
patience_counter = 0
BEST_MODEL_PATH = "best_vit5_aptos.pth"

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

EPOCHS_STAGE1 = 8
WARMUP_EPOCHS = 2

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS_STAGE1 - WARMUP_EPOCHS
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_EPOCHS]
)

In [ ]:
print("=== TAHAP 1: Linear Probing ===")
for epoch in range(EPOCHS_STAGE1):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1} — Learning Rate: {current_lr:.2e}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, EPOCHS_STAGE1)
    val_loss, val_acc, val_qwk = validate_one_epoch(model, val_loader, criterion, device, epoch, EPOCHS_STAGE1)

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)

    print(f"Val QWK: {val_qwk:.4f} | Val Acc: {val_acc:.4f} | Val MSE Loss: {val_loss:.4f}")

    if val_qwk > best_val_qwk: # Disimpan berdasarkan QWK Terbaik
        best_val_qwk = val_qwk
        best_val_loss = val_loss
        patience_counter = 0
        state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(state_dict, BEST_MODEL_PATH)
        print(f"--> Model terbaik disimpan! (Val QWK: {val_qwk:.4f})\n")
    # else:
    #     patience_counter += 1
    #     print(f"({patience_counter}/{patience})\n")
    #     if patience_counter >= patience:
    #         print(f"Early stopping Tahap 1 pada epoch {epoch+1}.")
    #         break

In [ ]:
# ===== TAHAP 2: Muat model terbaik dari Tahap 1, lalu buka 4 blok terakhir =====
model.module.load_state_dict(torch.load(BEST_MODEL_PATH))

for param in model.parameters():
    param.requires_grad = False

for i in range(5, 11):  # buka blok Transformers
    for param in model.module.blocks[i].parameters():
        param.requires_grad = True

for param in model.module.norm.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"[Tahap 2] Trainable: {trainable_params:,} / Total: {total_params:,} ({trainable_params/total_params*100:.2f}%)")

EPOCHS_STAGE2 = 40
WARMUP_EPOCHS_STAGE2 = 3

optimizer = optim.AdamW(model.parameters(), lr=8e-4, weight_decay=0.3)

warmup_scheduler2 = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS_STAGE2)
cosine_scheduler2 = CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2 - WARMUP_EPOCHS_STAGE2)
scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler2, cosine_scheduler2], milestones=[WARMUP_EPOCHS_STAGE2])

In [ ]:
patience_counter = 0  # reset patience untuk tahap baru

print("=== TAHAP 2: Fine-tuning 4 blok terakhir + norm ===")

torch.cuda.synchronize()  # Sinkronisasi GPU agar pencatatan waktu akurat
start_train_time = time.time()

for epoch in range(EPOCHS_STAGE2):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1} — Learning Rate: {current_lr:.2e}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, EPOCHS_STAGE2)
    val_loss, val_acc, val_qwk = validate_one_epoch(model, val_loader, criterion, device, epoch, EPOCHS_STAGE2)

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)

    print(f"Val QWK: {val_qwk:.4f} | Val Acc: {val_acc:.4f} | Val MSE Loss: {val_loss:.4f}")

    if val_qwk > best_val_qwk:
        best_val_qwk = val_qwk
        best_val_loss = val_loss
        patience_counter = 0
        state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(state_dict, BEST_MODEL_PATH)
        print(f"--> Model terbaik disimpan! (Val QWK: {val_qwk:.4f})\n")
    # else:
    #     patience_counter += 1
    #     print(f"({patience_counter}/{patience})\n")
    #     if patience_counter >= patience:
    #         print(f"Early stopping epoch {epoch+1}.")
    #         break

torch.cuda.synchronize()
end_train_time = time.time()

# 3. Hitung selisih waktu total
total_train_time = end_train_time - start_train_time

# Konversi detik ke format Menit:Detik agar mudah dibaca
minutes = int(total_train_time // 60)
seconds = int(total_train_time % 60)

print(f"\nTahap 2 selesai. Best Val Loss keseluruhan: {best_val_loss:.4f}")
print(f"\nTotal Waktu Training: {minutes} menit {seconds} detik ({total_train_time:.2f} detik)")

# Evaluasi

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'], label='Val Loss')
axes[0].set_title('Grafik Loss Training vs Validasi')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], label='Train Accuracy')
axes[1].plot(epochs_range, history['val_acc'], label='Val Accuracy')
axes[1].set_title('Grafik Akurasi Training vs Validasi')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves_vit5.png", dpi=150)
plt.show()

In [ ]:
model.eval()

all_test_preds, all_test_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = get_logits(model(images)).squeeze(-1)
        preds = torch.round(torch.clamp(outputs, 0, 4))

        all_test_preds.extend(preds.cpu().numpy())
        all_test_labels.extend(labels.cpu().numpy())

In [ ]:
label_names = {
    0: "No DR (Normal)",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR"
}
CLASS_NAMES = [label_names[i] for i in range(5)]

# --- Confusion Matrix ---
cm = confusion_matrix(all_test_labels, all_test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — ViT 5 (MSE Loss + Sampler)')
plt.xlabel('Prediksi')
plt.ylabel('Label Sebenarnya')
plt.tight_layout()
plt.savefig("confusion_matrix_vit5_mse.png", dpi=150)
plt.show()

report = classification_report(all_test_labels, all_test_preds, target_names=CLASS_NAMES, digits=4)
print("Classification Report (Data Testing):")
print(report)

# --- Metrik ringkasan ---
test_acc = accuracy_score(all_test_labels, all_test_preds)
test_qwk = cohen_kappa_score(all_test_labels, all_test_preds, weights="quadratic")

print(f"Akurasi pada Data Testing : {test_acc*100:.2f}%")
print(f"QWK pada Data Testing     : {test_qwk:.4f}")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, cohen_kappa_score

# ==========================================
# 1. KONVERSI LABEL MULTI-KELAS KE BINER
# ==========================================
# Kelas 0 tetap 0 (No DR). Kelas 1, 2, 3, 4 diubah menjadi 1 (DR)
all_test_labels_bin = np.where(np.array(all_test_labels) > 0, 1, 0)
all_test_preds_bin = np.where(np.array(all_test_preds) > 0, 1, 0)

BINARY_CLASS_NAMES = ["No DR (Normal)", "DR (Sakit)"]

# ==========================================
# 2. CONFUSION MATRIX BINER
# ==========================================
cm_bin = confusion_matrix(all_test_labels_bin, all_test_preds_bin)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_bin, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=BINARY_CLASS_NAMES, 
    yticklabels=BINARY_CLASS_NAMES,
    annot_kws={"size": 12}
)
plt.title('Binary Confusion Matrix — ViT 5 (MSE Loss + Sampler)', fontsize=11, fontweight='bold')
plt.xlabel('Prediksi', fontsize=10)
plt.ylabel('Label Sebenarnya', fontsize=10)
plt.tight_layout()
plt.savefig("binary_confusion_matrix_vit5_mse.png", dpi=150)
plt.show()

# ==========================================
# 3. CLASSIFICATION REPORT BINER
# ==========================================
report_bin = classification_report(
    all_test_labels_bin, 
    all_test_preds_bin, 
    target_names=BINARY_CLASS_NAMES, 
    digits=4
)
print("==================================================")
print("Classification Report Varian BINER (Data Testing):")
print("==================================================")
print(report_bin)

# ==========================================
# 4. METRIK RINGKASAN BINER
# ==========================================
test_acc_bin = accuracy_score(all_test_labels_bin, all_test_preds_bin)
# Catatan: Untuk kasus biner, QWK nilainya akan setara dengan Cohen's Kappa Linear standar
test_kappa_bin = cohen_kappa_score(all_test_labels_bin, all_test_preds_bin)

print(f"Akurasi Biner pada Data Testing : {test_acc_bin*100:.2f}%")
print(f"Kappa Biner pada Data Testing   : {test_kappa_bin:.4f}")


In [ ]:
model.module.load_state_dict(torch.load(BEST_MODEL_PATH))

model.eval()

all_test_preds, all_test_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = get_logits(model(images)).squeeze(-1)
        preds = torch.round(torch.clamp(outputs, 0, 4))

        all_test_preds.extend(preds.cpu().numpy())
        all_test_labels.extend(labels.cpu().numpy())

In [ ]:
label_names = {
    0: "No DR (Normal)",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR"
}
CLASS_NAMES = [label_names[i] for i in range(5)]

# --- Confusion Matrix ---
cm = confusion_matrix(all_test_labels, all_test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — ViT 5 (MSE Loss + Sampler)')
plt.xlabel('Prediksi')
plt.ylabel('Label Sebenarnya')
plt.tight_layout()
plt.savefig("confusion_matrix_vit5_mse.png", dpi=150)
plt.show()

report = classification_report(all_test_labels, all_test_preds, target_names=CLASS_NAMES, digits=4)
print("Classification Report (Data Testing):")
print(report)

# --- Metrik ringkasan ---
test_acc = accuracy_score(all_test_labels, all_test_preds)
test_qwk = cohen_kappa_score(all_test_labels, all_test_preds, weights="quadratic")

print(f"Akurasi pada Data Testing : {test_acc*100:.2f}%")
print(f"QWK pada Data Testing     : {test_qwk:.4f}")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, cohen_kappa_score

# ==========================================
# 1. KONVERSI LABEL MULTI-KELAS KE BINER
# ==========================================
# Kelas 0 tetap 0 (No DR). Kelas 1, 2, 3, 4 diubah menjadi 1 (DR)
all_test_labels_bin = np.where(np.array(all_test_labels) > 0, 1, 0)
all_test_preds_bin = np.where(np.array(all_test_preds) > 0, 1, 0)

BINARY_CLASS_NAMES = ["No DR (Normal)", "DR (Sakit)"]

# ==========================================
# 2. CONFUSION MATRIX BINER
# ==========================================
cm_bin = confusion_matrix(all_test_labels_bin, all_test_preds_bin)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_bin, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=BINARY_CLASS_NAMES, 
    yticklabels=BINARY_CLASS_NAMES,
    annot_kws={"size": 12}
)
plt.title('Binary Confusion Matrix — ViT 5 (MSE Loss + Sampler)', fontsize=11, fontweight='bold')
plt.xlabel('Prediksi', fontsize=10)
plt.ylabel('Label Sebenarnya', fontsize=10)
plt.tight_layout()
plt.savefig("binary_confusion_matrix_vit5_mse.png", dpi=150)
plt.show()

# ==========================================
# 3. CLASSIFICATION REPORT BINER
# ==========================================
report_bin = classification_report(
    all_test_labels_bin, 
    all_test_preds_bin, 
    target_names=BINARY_CLASS_NAMES, 
    digits=4
)
print("==================================================")
print("Classification Report Varian BINER (Data Testing):")
print("==================================================")
print(report_bin)

# ==========================================
# 4. METRIK RINGKASAN BINER
# ==========================================
test_acc_bin = accuracy_score(all_test_labels_bin, all_test_preds_bin)
# Catatan: Untuk kasus biner, QWK nilainya akan setara dengan Cohen's Kappa Linear standar
test_kappa_bin = cohen_kappa_score(all_test_labels_bin, all_test_preds_bin)

print(f"Akurasi Biner pada Data Testing : {test_acc_bin*100:.2f}%")
print(f"Kappa Biner pada Data Testing   : {test_kappa_bin:.4f}")
